# Setup

In [2]:
from albumentations import (
    Compose as ACompose,
    HorizontalFlip as AHorizontalFlip,
    VerticalFlip as AVerticalFlip
)

from torch.utils.data import Dataset, DataLoader

import os 
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
import ActivationExamples_SARSeg.examples_utils as utils
from ActivationExamples_SARSeg.examples_utils import *

In [ ]:
dataset_lmdb_path = '../data/BigEarthNet/Encoded-BigEarthNet'
parquet_path = '../data/BigEarthNet/metadata.parquet'
examples_lmdb_path = "../data/BigEarthNet/examples_subsample" # Alternatively go for all data

device = 'cuda' if torch.cuda.is_available() else 'cpu'

seed=24   # Fix the seed for reproducibility
batch_size= 16
NUM_CLASSES = 20 # 19 classes + 1 unlabeled
n_classes = num_class = NUM_CLASSES

IMG_HEIGHT = 120
IMG_WIDTH  = 120
IMG_CHANNELS = 2

## Train, Val, Test Split

In [ ]:
# Define the transformations
transformx = ACompose([
    AHorizontalFlip(p=0.5),
    AVerticalFlip(p=0.5),
],is_check_shapes=False)

# Parquet file with metadata for Big Earth dataset
train_matches, val_matches, test_matches = match_keys(parquet_path)

print(f"Train size: {len(train_matches)}, Val size: {len(val_matches)}, Test size: {len(test_matches)}")

# Create datasets
train_dataset = SARSegmentationDataset120(dataset_lmdb_path, train_matches, transform=transformx)
val_dataset = SARSegmentationDataset120(dataset_lmdb_path, val_matches, transform=None)
test_dataset = SARSegmentationDataset120(dataset_lmdb_path, test_matches, transform=None)

# Create data loaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Define the model metrcis and load model
num_train_img = len(train_dataset)
num_val_img = len(val_dataset)
num_test_img = len(test_dataset)

steps_per_epoch = num_train_img//batch_size
val_steps_per_epoch = num_val_img//batch_size
print("Steps per epoch: ", steps_per_epoch)
print("Validation steps per epoch: ", val_steps_per_epoch)


Train size: 237871, Val size: 122342, Test size: 119825
Opening LMDB environment ...
Opening LMDB environment ...
Opening LMDB environment ...
Steps per epoch:  14866
Validation steps per epoch:  7646


# Subsample and Generate Examples and Store in LMDB

In [5]:
# Load the model
model = load_from_checkpoint_skipconn("../models/unet120_full_skipconn_best.pth")
model.eval();
model.to(device)

to_save_layers = ["encoder.layer3", "encoder.layer4", "decoder.up1", "decoder.up2"]

random.seed(42)
train_matches_subsample = random.sample(train_matches, len(train_matches)//10)

subsample_image_keys = []
subsample_mask_keys = []

for match in train_matches_subsample:
    train_image_key, train_mask_key = match
    subsample_image_keys.append(train_image_key)
    subsample_mask_keys.append(train_mask_key)

print(train_matches_subsample[:5])

CustomUnet initialized with encoder channels: [2, 64, 256, 512, 1024, 2048]
[('S1A_IW_GRDH_1SDV_20180126T161302_34VDN_15_88', 'S2B_MSIL2A_20180127T102259_N9999_R065_T34VDN_15_88_reference_map'), ('S1A_IW_GRDH_1SDV_20170903T153258_35VNL_70_4', 'S2A_MSIL2A_20170905T095031_N9999_R079_T35VNL_70_04_reference_map'), ('S1A_IW_GRDH_1SDV_20170617T064659_29UPU_62_3', 'S2A_MSIL2A_20170617T113321_N9999_R080_T29UPU_62_03_reference_map'), ('S1A_IW_GRDH_1SDV_20180420T063917_29UPU_27_80', 'S2B_MSIL2A_20180421T114349_N9999_R123_T29UPU_27_80_reference_map'), ('S1A_IW_GRDH_1SDV_20180413T161957_34UEG_40_3', 'S2A_MSIL2A_20180413T095031_N9999_R079_T34UEG_40_03_reference_map')]


In [6]:
want_to_store = False # Set to True if you want to store the activations
store_activations(train_matches_subsample, model, examples_lmdb_path, dataset_lmdb_path, to_save_layers, break_flag = not(want_to_store))

Storing activations in LMDB at path: ../data/BigEarthNet/examples_subsample


  0%|          | 0/23787 [00:00<?, ?it/s]


## Testing the stored Examples to be correctly extractable

In [ ]:
random.seed(None)
random_index = random.randint(0, len(train_matches_subsample) - 1)
image_reference = train_matches_subsample[random_index][0]
random_layer_index = random.randint(0, len(to_save_layers) - 1)
layer_name = to_save_layers[random_layer_index]
test_store_activations(model, examples_lmdb_path, dataset_lmdb_path, image_reference, layer_name)

Loaded activation shape: torch.Size([1, 256, 8, 8])
Activations match successfully!
